# 01.6 · 可观测性（Metrics + Trace）

> 为 Multi-Agent 系统添加指标与链路追踪。


In [ ]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

---
## Part 6 · 可观测性（Metrics + Trace）

生产中用 Prometheus + OpenTelemetry，这里用纯 Python 模拟相同概念：
- **Metrics**：计数器、直方图（延迟分布）
- **Trace**：全链路 request_id，记录每个 Agent 的耗时与结果

In [ ]:
import statistics

# ── 简易 Metrics 收集器 ────────────────────────────────────────────────────
class Metrics:
    def __init__(self):
        self.counters: dict[str, int]       = defaultdict(int)
        self.histograms: dict[str, list]    = defaultdict(list)

    def inc(self, name: str, labels: dict = {}):
        key = name + str(sorted(labels.items()))
        self.counters[key] += 1

    def observe(self, name: str, value: float, labels: dict = {}):
        key = name + str(sorted(labels.items()))
        self.histograms[key].append(value)

    def summary(self):
        print("\n📊 Metrics Summary")
        print("─" * 50)
        for k, v in self.counters.items():
            print(f"  counter  {k} = {v}")
        for k, vals in self.histograms.items():
            p50 = statistics.median(vals)
            p95 = sorted(vals)[int(len(vals)*0.95)] if len(vals) > 1 else vals[0]
            print(f"  hist     {k}  p50={p50:.3f}s  p95={p95:.3f}s  n={len(vals)}")

metrics = Metrics()

# ── 带 Trace 和 Metrics 的 Pipeline ─────────────────────────────────────────
def traced_pipeline(query: str) -> dict:
    trace_id = str(uuid.uuid4())[:8]
    trace_log = []

    def span(name: str, fn, *args, **kwargs):
        t = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = round(time.perf_counter() - t, 3)
        trace_log.append({"span": name, "latency_s": elapsed})
        metrics.observe("agent_latency_seconds", elapsed, {"agent": name})
        metrics.inc("agent_requests_total", {"agent": name, "status": "ok"})
        return result

    hits    = span("retriever", fake_retrieve, query, latency=0.3)
    topk    = span("reranker",  fake_rerank,   query, hits, latency=0.2)
    answer  = span("generator", fake_generate, query, topk, latency=0.5)
    verdict = span("verifier",  fake_verify,   answer, topk, latency=0.15)

    print(f"\n🔍 Trace [{trace_id}] for: {query[:30]}")
    for s in trace_log:
        bar = "█" * int(s["latency_s"] * 20)
        print(f"  {s['span']:<12} {bar:<20} {s['latency_s']}s")

    return {"answer": answer, "verdict": verdict, "trace_id": trace_id}

# ── 跑 5 次，生成 metrics ────────────────────────────────────────────────────
for q in ["RAG 是什么？", "幂等设计？", "可观测性？", "Blackboard？", "Pipeline？"]:
    traced_pipeline(q)

metrics.summary()

## 📖 讲义

# Part 6
## 课堂练习 & 总结

---

# 课堂练习（五个阶段）

| 阶段 | 任务 | 目标 |
|------|------|------|
| **1. 入门** | 实现 Redis Pipeline 三 Agent（Retriever → Generator → Verifier），跑通端到端并打印 trace | 理解消息传递基础 |
| **2. 进阶** | 把 Reranker 抽成独立 HTTP 服务，用 Docker Compose 编排，压测并观察延迟 | 掌握微服务拆分 |
| **3. 监控** | 为每个 Agent 加 Prometheus 指标 + p95 告警，演练 Reranker 下线降级 | 生产可观测性 |
| **4. 安全** | 实现 Pydantic 入参校验 + 参数白名单，模拟恶意参数并记录拒绝事件 | 安全意识 |
| **5. 评估** | 接入 RAGAS 运行 `rag_e2e_eval.py`，保存多次版本的评估结果对比 | 质量迭代意识 |

---

# 常见陷阱速查

| 陷阱 | 典型症状 | 对策 |
|------|----------|------|
| 单点瓶颈 | Orchestrator CPU 100%，其他 Agent 空闲 | 拆分职责，引入异步队列 |
| 共享可变状态 | 随机数据错乱，难以复现 | 乐观锁 / 事件溯源 |
| 缺少幂等 | 重试导致重复写入或重复 API 调用 | request-id + 处理记录 |
| 无监控 | 出问题后无法定位根因 | Metrics + Traces + 结构化日志 |
| 无限循环 | 资源耗尽，成本暴增 | 最大调用深度 + 成本预算 |
| 过度信任 LLM | 执行了恶意 tool call | 参数白名单 + 人工审批 |
| 忽视隐私 | 敏感数据在多 Agent 间明文传播 | PII 脱敏 + 访问控制 |

---

# 课程小结

**Multi-Agent Architecture 的核心价值**

```
复杂任务 ──拆分──▶ 多个职责明确的 Agent ──协作──▶ 可扩展的生产系统
```

**三个必记原则**
1. **职责分离 + 松耦合** — Agent 之间只通过接口（消息/API）交互
2. **幂等 + 可观测** — 每条消息有 ID，每个 Agent 暴露 metrics/traces
3. **降级 + 安全** — 提前设计失败路径，永不直接执行 LLM 输出

**技术演进路径**
```
单体 Agent
  → Pipeline（Redis Queue）
    → Hub-and-Spoke（HTTP 微服务）
      → Blackboard（事件驱动）
        → 混合架构（按需组合）
```

---

# 推荐资源

<div class="columns">
<div>

### 框架 & 工具
- **LangChain / LangGraph** — Multi-Agent patterns
- **CrewAI** — Role-based agents
- **AutoGen** — Microsoft multi-agent framework
- **Haystack** — Production RAG pipelines

### 消息队列
- **Redis Streams** — 入门首选
- **RabbitMQ** — 经典 AMQP
- **Apache Kafka** — 高吞吐场景

</div>
<div>

### 可观测性
- **LangSmith / Langfuse** — LLM tracing
- **OpenTelemetry** — 标准 traces
- **Prometheus + Grafana** — Metrics

### 部署
- **Docker Compose** — 本地多服务
- **Kubernetes + HPA** — 生产弹性扩缩
- **Pinecone / Qdrant / OpenSearch** — 向量数据库

### 评估
- **RAGAS** — RAG 质量评估框架

</div>
</div>

---

---
**导航**：[← 01.5 · 幂等设计](./01.5_idempotency.ipynb) · [📋 课程索引](./01_multi_agent_arch_demo.ipynb) · [01.7 · 优雅降级 →](./01.7_graceful_degradation.ipynb)
